# IE-OPT-04 · 仓储装箱与车次配置（原创 PySCIPOpt 教学案例）

**工业情境**：一组不同体积的零部件待装载到相同容量的物流箱中，目标是使用尽可能少的箱子。课堂数据是假设性教学案例。

参考阅读：[PySCIPOpt 官方 bpp.py](https://github.com/scipopt/PySCIPOpt/blob/master/examples/finished/bpp.py)，本站代码为独立实现。官方文件展示 FFD 与扩展建模思路，学生应核对假设，勿把第三方代码复制当成自己的实现。

## 1. 数学模型

- y_b ∈ {0,1} 表示箱 b 是否启用。
- x_ib ∈ {0,1} 表示物品 i 是否放入箱 b。
- 最小化 Σ_b y_b。
- 每件物品恰好装入一个箱；每箱体积不得超过容量乘以启用变量。
- 加入 y_b ≥ y_(b+1) 消除明显对称性。

In [ ]:
from pyscipopt import Model, quicksum

sizes = [4, 3, 3, 2, 2, 1, 1]  # 合成物品尺寸
capacity = 6
assert all(0 < x <= capacity for x in sizes)

In [ ]:
def solve_bins(sizes, capacity):
    n = len(sizes)
    model = Model("IE-OPT-04 bin packing")
    model.hideOutput(True)
    y = [model.addVar(vtype="B", name=f"box_{b}") for b in range(n)]
    x = {(i, b): model.addVar(vtype="B", name=f"assign_{i}_{b}")
         for i in range(n) for b in range(n)}
    for i in range(n):
        model.addCons(quicksum(x[i, b] for b in range(n)) == 1)
    for b in range(n):
        model.addCons(quicksum(sizes[i] * x[i, b] for i in range(n))
                      <= capacity * y[b])
    for b in range(n - 1):
        model.addCons(y[b] >= y[b + 1])
    model.setObjective(quicksum(y), "minimize")
    model.optimize()
    assert str(model.getStatus()) == "optimal"
    assignment = [next(b for b in range(n) if model.getVal(x[i, b]) > .5)
                  for i in range(n)]
    return {"bins": int(round(model.getObjVal())), "assignment": assignment}

answer = solve_bins(sizes, capacity)
print("最少箱数:", answer["bins"])
print("物品对应箱号:", answer["assignment"])

## 2. 独立穷举 Oracle

采用递归装箱，每次只往已经使用的箱子中放入物品或开一个新箱，不调用 SCIP。相同剩余容量的箱子视为对称，避免重复搜索。

In [ ]:
def oracle_min_bins(sizes, capacity):
    best = len(sizes)
    loads = []
    def dfs(i):
        nonlocal best
        if i == len(sizes):
            best = min(best, len(loads))
            return
        if len(loads) >= best:
            return
        observed = set()
        for b, used in enumerate(loads):
            if used in observed or used + sizes[i] > capacity:
                continue
            observed.add(used)
            loads[b] += sizes[i]
            dfs(i + 1)
            loads[b] -= sizes[i]
        if len(loads) + 1 < best:
            loads.append(sizes[i])
            dfs(i + 1)
            loads.pop()
    dfs(0)
    return best

oracle = oracle_min_bins(sizes, capacity)
assert answer["bins"] == oracle, (answer, oracle)
assert all(sum(sizes[i] for i, b in enumerate(answer["assignment"]) if b == j)
           <= capacity for j in set(answer["assignment"]))
assert len(set(answer["assignment"])) == answer["bins"]
print("独立验证通过，最优箱数 =", oracle)

## 3. 工程扩展与课后任务

1. 改变箱容量，记录箱数变化与货箱空闲率。
2. 增加车辆载重限制、物品不相容性或优先发运规则；讨论模型如何扩展。
3. 实现 FFD 启发式，与小规模最优解公平比较，报告相对差距。
4. 解释这里的静态装箱最优为何不能代替有随机到达和交通约束的真实仓储物流仿真。

**该 Notebook 是本地教学示例，未接入正式 L2 判分。**